# [M4] AI NHÌN VÀ HIỂU DỮ LIỆU
## Buổi S01: Biểu Diễn Ảnh Số: Pixel, Kênh Màu (Grayscale/RGB) & Image Tensors

---

### Metadata Khóa Học
* **Module:** M4 — AI NHÌN VÀ HIỂU DỮ LIỆU
* **Buổi:** S01
* **Thời lượng dự kiến:** 120 phút
* **Sản phẩm đầu ra (Session Output):** Notebook xử lý và trực quan hóa Tensor ảnh số chuẩn PyTorch
* **Mức độ liên quan VOAI / Olympic AI:** Nền tảng của toàn bộ nhánh Computer Vision trong các kỳ thi Olympic AI.

#### Mục tiêu học tập (Learning Objectives)
- Hiểu bản chất số học của ảnh số: Điểm ảnh (Pixels) là các giá trị cường độ sáng trong thang đo [0, 255].
- Phân biệt cấu trúc tensor của ảnh đa kênh: Grayscale [H, W] vs RGB [H, W, C] vs PyTorch [C, H, W].
- Thực hiện chuẩn hóa điểm ảnh (Normalization) từ [0, 255] về khoảng [0.0, 1.0].
- Bóc tách từng kênh màu (Red, Green, Blue) và trực quan hóa biểu đồ phân bố cường độ sáng (Intensity Histogram).

#### Điều kiện tiên quyết (Prerequisites)
- Hoàn thành Module M3 (Đã làm chủ PyTorch Tensors).
- Kỹ năng sử dụng Matplotlib và NumPy.

---


## 1. WHY? — TẠI SAO PHẢI HỌC CHỦ ĐỀ NÀY?


Máy tính không bao giờ "nhìn thấy" một chú mèo đáng yêu hay một chiếc ô tô đang di chuyển. Thứ duy nhất máy tính nhìn thấy là một **Ma trận số khổng lồ (2D hoặc 3D Matrix)** chứa hàng triệu con số nguyên. Nếu bạn không hiểu cách biểu diễn này, bạn sẽ liên tục gặp lỗi lệch thứ tự kênh màu (ví dụ OpenCV dùng BGR trong khi Matplotlib/PyTorch dùng RGB), hoặc đưa các giá trị pixel thô từ 0 đến 255 vào mạng nơ-ron khiến mô hình bị bão hòa gradient ngay lập tức.

Làm chủ biểu diễn ảnh số là viên gạch đầu tiên để bạn bước vào thế giới Thị giác Máy tính (Computer Vision) — lĩnh vực chiếm tỷ trọng lớn trong các đề thi Olympic AI cấp quốc gia và quốc tế.



## 2. CONCEPT — NỀN TẢNG LÝ THUYẾT & TRỰC GIÁC


### 1. Bản chất của Ảnh số (Digital Image Representation)
* Một bức ảnh kỹ thuật số là một lưới các điểm ảnh (Pixels). Mỗi pixel lưu trữ độ sáng.
* **Ảnh xám (Grayscale):** Là một ma trận 2 chiều có kích thước $Height \times Width$. Mỗi giá trị là một số nguyên từ 0 (Đen tuyệt đối) đến 255 (Trắng tuyệt đối).
* **Ảnh màu (RGB):** Là một mảng 3 chiều gồm 3 ma trận màu xếp chồng lên nhau: **R** (Red - Đỏ), **G** (Green - Lục), **B** (Blue - Lam).

### 2. Sự khác biệt về Thứ tự Chiều (Dimension Ordering)
* **Quy ước của Thư viện Ảnh (PIL / OpenCV / Matplotlib):** Định dạng **$[H, W, C]$** (Chiều cao, Chiều rộng, Số kênh màu).
* **Quy ước bắt buộc của PyTorch:** Định dạng **$[C, H, W]$** (Số kênh màu đứng trước: Channel-First), hoặc khi nạp theo batch là **$[Batch, C, H, W]$**.
* *Chuyển đổi trong PyTorch:* Dùng `tensor.permute(1, 2, 0)` để đổi từ $[C, H, W] \to [H, W, C]$ khi cần hiển thị bằng Matplotlib!

### 3. Chuẩn hóa Điểm ảnh (Pixel Normalization)
Đưa các giá trị nguyên $[0, 255]$ về số thực $[0.0, 1.0]$ bằng cách chia cho $255.0$. Thao tác này giúp các thuật toán tối ưu hóa Gradient Descent hội tụ nhanh và ổn định hơn rất nhiều.



## 3. VISUAL INTUITION — TRỰC QUAN HÓA KIẾN THỨC


```
Cấu trúc Tensor Ảnh RGB:
               Height (H)
                │
                ▼   Width (W)
      ┌────────────┬────────────┐
   R  │  Pixel R   │  Pixel R   │ ──> Kênh Đỏ (Red Channel)
   ┌──┴────────────┴────────────┤
G  │    Pixel G    │  Pixel G   │ ──> Kênh Lục (Green Channel)
┌──┴────────────┬───┴───────────┤
│    Pixel B    │    Pixel B    │ ──> Kênh Lam (Blue Channel)
└───────────────┴───────────────┘
  PyTorch Tensor Shape: [3, H, W]
```



## 4. CODE IMPLEMENTATION & 5. WALKTHROUGH

*Toàn bộ code dưới đây được thiết kế thực tế, có thể chạy độc lập không phụ thuộc tải dữ liệu bên ngoài.*

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# 1. Tu tao mot anh mau RGB nhan tao 8x8 pixels de quan sat ro ban chat tung con so
np.random.seed(42)

# Tao ma tran anh RGB kich thuoc (8, 8, 3) voi cac vung mau dac trung
synthetic_img = np.zeros((8, 8, 3), dtype=np.uint8)
synthetic_img[:4, :4, 0] = 255   # O vuong tren-trai mau Do (R=255, G=0, B=0)
synthetic_img[:4, 4:, 1] = 255   # O vuong tren-phai mau Luc (R=0, G=255, B=0)
synthetic_img[4:, :4, 2] = 255   # O vuong duoi-trai mau Lam (R=0, G=0, B=255)
synthetic_img[4:, 4:, :] = 255   # O vuong duoi-phai mau Trang (R=255, G=255, B=255)

print("===== MA TRAN ANH RGB (8x8x3) =====")
print(f"Shape anh goc: {synthetic_img.shape} | Kieu du lieu: {synthetic_img.dtype}")
print(f"Gia tri pixel tai goc tren-trai (Kenh R): {synthetic_img[0, 0, 0]}")

# 2. Chuyen doi sang PyTorch Tensor theo chuan [C, H, W] va Chuan hoa ve [0.0, 1.0]
tensor_img = torch.from_numpy(synthetic_img).permute(2, 0, 1).float() / 255.0
print(f"Shape PyTorch Tensor sau permute: {tensor_img.shape} | Min: {tensor_img.min()}, Max: {tensor_img.max()}")

### 🔍 Code Walkthrough (Khối 1)
Khối code tạo dựng một bức ảnh số nhân tạo nhỏ gọn:
- Giúp người học nhìn thấu từng con số pixel trong ma trận $8 	imes 8 	imes 3$.
- Sử dụng `.permute(2, 0, 1)` để đổi trục từ $[H, W, C]$ sang chuẩn $[C, H, W]$ của PyTorch.
- Thực hiện chuẩn hóa chia $255.0$ đưa miền giá trị về $[0.0, 1.0]$ an toàn số học.


In [ ]:
# 3. Tach 3 kenh mau va Truc quan hoa bang Matplotlib
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Hien thi anh mau tong hop (can chuyen lai ve [H, W, C] de Matplotlib hieu)
img_to_show = tensor_img.permute(1, 2, 0).numpy()
axes[0].imshow(img_to_show)
axes[0].set_title('Anh Mau RGB Tong Hop', fontweight='bold')
axes[0].axis('off')

# Kenh Do (Red)
axes[1].imshow(tensor_img[0].numpy(), cmap='Reds')
axes[1].set_title('Kenh Do (R)', fontweight='bold')
axes[1].axis('off')

# Kenh Luc (Green)
axes[2].imshow(tensor_img[1].numpy(), cmap='Greens')
axes[2].set_title('Kenh Luc (G)', fontweight='bold')
axes[2].axis('off')

# Kenh Lam (Blue)
axes[3].imshow(tensor_img[2].numpy(), cmap='Blues')
axes[3].set_title('Kenh Lam (B)', fontweight='bold')
axes[3].axis('off')

plt.tight_layout()
plt.show()

### 🔍 Code Walkthrough (Khối 2)
Khối code bóc tách trực quan 3 kênh màu Red, Green, Blue:
- Minh họa rõ ràng nguyên lý phối màu cộng (Additive Color): Khi cả 3 kênh cùng đạt cực đại 255, ta thu được màu trắng ở góc dưới bên phải.


## 6. PRACTICE — BÀI TẬP THỰC HÀNH


### Bài 1 (Easy): Chuyển ảnh màu sang ảnh xám (RGB to Grayscale)
Sử dụng công thức phối màu trọng số chuẩn mắt người:
$$	ext{Gray} = 0.299 R + 0.587 G + 0.114 B$$
Áp dụng công thức trên lên `tensor_img` để tạo ra một tensor ảnh xám kích thước `(1, 8, 8)`. Hiển thị bằng `cmap='gray'`.

### Bài 2 (Easy): Cắt lát không gian ảnh (Spatial Cropping)
Trích xuất góc phần tư phía trên bên phải của bức ảnh nhân tạo bằng cú pháp slicing PyTorch. In ra kích thước của ảnh sau khi cắt.

### Bài 3 (Medium): Vẽ biểu đồ phân bố cường độ sáng (Color Histogram)
Tạo một bức ảnh ngẫu nhiên kích thước $(64, 64, 3)$ với các giá trị từ 0 đến 255. Sử dụng Matplotlib vẽ 3 đường biểu đồ tần suất (Histograms) tương ứng với 3 kênh R, G, B để quan sát phân phối độ sáng.

### Bài 4 (Medium): Đảo ngược màu ảnh (Invert Colors)
Viết một hàm `invert_image(tensor)` nhận vào tensor ảnh đã chuẩn hóa $[0, 1]$ và thực hiện phép biến đổi âm bản (Negative image): $x_{inv} = 1.0 - x$. Hiển thị kết quả so sánh.

### Bài 5 (Challenge): Phát hiện thứ tự kênh BGR của OpenCV
Nêu rõ cách nhận biết một bức ảnh đang bị hiển thị sai thứ tự màu (ví dụ khuôn mặt người bị chuyển sang màu xanh da trời do thư viện đọc bằng BGR nhưng hiển thị bằng RGB). Viết một dòng lệnh PyTorch để lật ngược thứ tự kênh màu từ BGR sang RGB.



## 7. VOAI CONNECTION — CHIẾN LƯỢC PHÒNG THI OLYMPIC


Trong đề thi Olympic AI phần Thị giác Máy tính, một trong những lỗi gây mất điểm ngớ ngẩn nhất là **lỗi thứ tự kênh màu và thang đo pixel**:
Nhiều thí sinh nạp ảnh bằng thư viện OpenCV (`cv2.imread()`) nhưng quên rằng OpenCV đọc theo chuẩn `BGR` với thang $[0, 255]$, rồi nạp thẳng vào mô hình được huấn luyện trên chuẩn `RGB` với thang $[0.0, 1.0]$. Kết quả là mô hình hoàn toàn mù tịt, dự đoán sai lệch 100%. Nắm vững cách quản lý Tensor ảnh giúp bạn không bao giờ rơi vào cái bẫy này.



## 8. DEBUGGING CHALLENGE — BẮT LỖI THỰC CHIẾN


**Tình huống thực tế:** Thí sinh đưa trực tiếp một tensor PyTorch có kích thước `[3, 64, 64]` vào hàm `plt.imshow()`, hệ thống báo lỗi:

```python
# CODE CÓ LỖI CHANNEL-FIRST KHI HIỂN THỊ:
# plt.imshow(tensor_img) # Bao loi: TypeError: Invalid shape (3, 8, 8) for image data
```

**Nhiệm vụ của bạn:**
1. Giải thích tại sao Matplotlib chỉ chấp nhận ảnh 2D `[H, W]` hoặc 3D dạng `[H, W, C]`.
2. Sửa lỗi bằng cách sử dụng phương thức `.permute(1, 2, 0)` và chuyển sang mảng NumPy.



## 9. KNOWLEDGE CHECK — KIỂM TRA ĐỘ HIỂU BẢN CHẤT


1. Kích thước (Shape) chuẩn mực của một Tensor ảnh màu đơn lẻ trong PyTorch là gì?
   - A. `[Height, Width, Channels]`
   - B. `[Channels, Height, Width]`
   - C. `[Height, Channels, Width]`
   - D. `[Width, Height]`

2. Mục đích chính của việc chia tất cả các giá trị pixel cho 255.0 là gì?
   - A. Để giảm dung lượng lưu trữ trên đĩa cứng.
   - B. Chuẩn hóa dải giá trị từ số nguyên $[0, 255]$ về số thực $[0.0, 1.0]$, giúp các hàm kích hoạt và thuật toán Gradient Descent hoạt động ổn định, tránh bão hòa gradient.
   - C. Để chuyển ảnh màu thành ảnh xám.
   - D. Để tăng độ tương phản của ảnh.

3. Một bức ảnh xám có kích thước $100 	imes 100$ chứa bao nhiêu điểm ảnh (pixels)?
   - A. 100
   - B. 10,000
   - C. 30,000
   - D. 1,000,000

4. Lệnh `tensor.permute(1, 2, 0)` có tác dụng gì đối với một tensor có shape `(C, H, W)`?
   - A. Xóa kênh màu đầu tiên.
   - B. Hoán vị các trục để đưa tensor về shape `(H, W, C)`.
   - C. Xoay bức ảnh một góc 90 độ.
   - D. Nhân đôi số kênh màu.

5. Trong không gian màu RGB, khi một điểm ảnh có giá trị $(R=0, G=0, B=0)$, điểm ảnh đó sẽ hiển thị màu gì?
   - A. Màu Trắng
   - B. Màu Đỏ
   - C. Màu Đen tuyệt đối
   - D. Màu Vàng



## 10. SESSION SUMMARY — TỔNG KẾT BUỔI HỌC


* **What I learned:**
  - Bản chất số học của ảnh số: Điểm ảnh (Pixel) và các kênh màu.
  - Sự khác biệt giữa định dạng `[H, W, C]` của Matplotlib và `[C, H, W]` của PyTorch.
  - Tầm quan trọng của việc chuẩn hóa điểm ảnh về khoảng $[0.0, 1.0]$.
* **What I can now do:**
  - Tự tin thao tác, bóc tách và biến đổi ma trận ảnh số bằng PyTorch Tensor.
  - Hoán vị các chiều an toàn bằng `.permute()` để hiển thị và tiền xử lý.
* **What I still need to practice:**
  - Tìm hiểu cách tăng cường dữ liệu: **Data Augmentation** ở Buổi 2.



## 11. NEXT SESSION — BƯỚC ĐI TIẾP THEO


Trong **Buổi M4_S02**, chúng ta sẽ nâng cao kỹ năng xử lý ảnh: **Tiền Xử Lý Ảnh & Kỹ Thuật Tăng Cường Dữ Liệu (Data Augmentation)** với `torchvision.transforms` — Khám phá cách xoay, lật, cắt ảnh để nhân bản dữ liệu và giúp mô hình nhìn nhận thế giới từ nhiều góc độ!



---

## 🎯 Syllabus Traceability (Đối chiếu Syllabus)

Buổi học này đóng góp trực tiếp vào mục tiêu của **M4**:
- Biểu diễn ảnh số & ma trận điểm ảnh (Pixels) ✓
- Cấu trúc tensor ảnh Grayscale vs RGB ([C, H, W]) ✓
- Chuẩn hóa điểm ảnh & tách kênh màu ✓

* **Session Deliverable:** Mã nguồn xử lý tensor hình ảnh và bóc tách kênh màu chuẩn PyTorch
* **Đảm bảo chuẩn Syllabus Lock:** Nội dung nằm đúng chuẩn 100%, không lan man, giải quyết trực tiếp năng lực cốt lõi.
